In [ ]:
                    # Why Runnables?
# automate mutliple processes using chains
# Problems was soo many chains
# Learning Curve increase so was CodeBase
# Compatabible Issue with diff components as they were not standardise
# Runnable ( input --> process --> output )
# Have common interface (Invoke() , Batch() , Stream())
# Seamless Connection ( for complex workflow --> and will get return as Runnables only )

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

c:\Users\singh\Let's Gooooo\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Mahatma Gandhi is widely considered the "Father of India." This title was bestowed upon him due to his pivotal role in India\'s independence movement through non-violent civil disobedience.'

In [2]:
# Exmaple of LLM Application
promt = PromptTemplate(
    template= "give me a intresting Vlog about {topic}",
    input_variables=['topic']   
)

topic = input("Give me a topic")
parser = StrOutputParser()

chain = promt | llm_gemini | parser

chain.invoke(topic)

'Okay, here\'s an idea for a cricket vlog that aims to be interesting and engaging, combining on-field action with a personal narrative:\n\n**Title:** **"Chasing My Dream: From Backyard Cricket to the Big League? (ft. [Local Cricket Club/Team])"**\n\n**Vlog Concept:** This vlog follows a cricket enthusiast\'s journey to improve their skills and potentially join a local cricket team/club. It blends practice sessions, game footage, analysis, and personal reflections on the challenges and joys of pursuing a cricket dream.\n\n**Elements to Include:**\n\n*   **Introduction (0:00 - 0:30):**\n    *   Start with an energetic montage of cricket highlights (both professional and amateur - even some funny backyard cricket clips).\n    *   Introduce yourself and your passion for cricket.\n    *   Clearly state your goal: "This vlog is about my journey to become a better cricketer. I\'m aiming to join [Local Cricket Club/Team Name] and see how far I can go."\n    *   Set the scene: "I\'ve been play

In [ ]:
# First approch that Langchain took
class LLM:
    def __init__(self):
        print("LLM Invoked")
        
    def predict(self , prompt):
        return {"responce":f"the ans of the prompt {prompt} is Really Easy to give"}
    
class PromptTemplate():
    def __init__(self , template , input_varibale):
        self.template = template
        self.input_varibale = input_varibale
        
    def format(self , input_dict):
         return self.template.format(**input_dict)
  
temp_template = PromptTemplate(
    template= "Give me a {length} detailed blog about {topic}",
    input_varibale= ['length' , 'topic' ]
)   

# Tradition FLow
temp_llm = LLM()
promt = temp_template.format({'topic' : "india" , 'length' : "100 words long"})
temp_llm.predict(promt)

LLM Invoked


{'responce': 'the ans of the prompt Give me a 100 words long detailed blog about india is Really Easy to give'}

In [ ]:
# New Flow, Solving using LLMChain ( was not flexible )
class LLMChain:
    def __init__(self , llm , prompt):
        self.llm = llm
        self.prompt = prompt
        
    def run(self , input_dict):
        template = self.prompt.format(input_dict)
        result = self.llm.predict(template)
        
        return result['responce']
    
temp_llm_chain = LLMChain(llm = temp_llm , prompt= temp_template)

temp_llm_chain.run({'topic' : "india" , 'length' : "100 words long"})
        

'the ans of the prompt Give me a 100 words long detailed blog about india is Really Easy to give'

In [19]:
# Runnable Approch 
from abc import ABC, abstractmethod

class Runnable(ABC):
    @abstractmethod
    def invoke(input_date):
        pass

class LLM(Runnable):
    def __init__(self):
        print("LLM Invoked")
        
    def invoke(self , prompt):
        return {"responce":f"the ans of the prompt {prompt} is Really Easy to give"}
    
class PromptTemplate(Runnable):
    def __init__(self , template , input_varibale):
        self.template = template
        self.input_varibale = input_varibale
        
    def invoke(self , input_dict):
         return self.template.format(**input_dict)
     
class StrOutputParser:
    def __init__(self):
        pass
        
    def invoke(self , input_data):
        return input_data['responce']
     
class RunnableConnector(Runnable):
    def __init__(self , runnable_list):
        self.runnable_list = runnable_list
        
    def invoke(self , input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
            
        return input_data
    
temp_template = PromptTemplate(
    template= "Give me a {length} detailed blog about {topic}",
    input_varibale= ['length' , 'topic' ]
)  

temp_llm = LLM()

parser = StrOutputParser()

temp_runnable = RunnableConnector([temp_template , temp_llm , parser])
temp_runnable.invoke({'topic' : "india" , 'length' : "100 words long"})


LLM Invoked


'the ans of the prompt Give me a 100 words long detailed blog about india is Really Easy to give'

In [ ]:
promt_template1 = PromptTemplate(
    template= "make a joke about following \n {topic}",
    input_varibale=['topic']
)
promt_template2 = PromptTemplate(
    template= "Explain the following Joke in detail \n {joke}",
    input_varibale=['joke']
)

chain1 = RunnableConnector([promt_template1 , temp_llm ])
chain2 = RunnableConnector([promt_template2 , temp_llm , parser])
final_chain = RunnableConnector([chain1 , chain2])
final_chain.invoke({'topic' : "Ai"})